# plate solving using astrometry.net

## 필요한 환경

이 프로젝트를 위해서는 아래의 모듈이 필요합니다.

> numpy, matplotlib, astropy, version_information


### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [ ]:
import importlib, sys, subprocess
packages = "numpy, matplotlib, astropy, version_information" # required modules
pkgs = packages.split(", ")

for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed... now start install")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"****** module {pkg} is installed")
    else: 
        print(f"**** module {pkg} is installed")

%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

### import modules

In [ ]:
from glob import glob
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData
from astroquery.astrometry_net import AstrometryNet

import ysfitsutilpy as yfu

import _astro_utilities
import _Python_utilities

# 환경 설정


In [ ]:
#%%
#######################################################
# read all files in base directory for processing
BASEDIR = Path("/mnt/Rdata/OBS_data") 
DOINGDIR = Path(BASEDIR/ "2024-EXO" / "RiLA600_STX-16803_-_1bin")
DOINGDIR = Path(BASEDIR/ "2024-EXO" / "GSON300_STF-8300M_-_1bin")

DOINGDIRs = sorted(_Python_utilities.getFullnameListOfsubDirs(DOINGDIR))
DOINGDIRs = sorted([x for x in DOINGDIR.iterdir() if x.is_dir()])
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

mas1 = [x for x in DOINGDIRs if "CAL-BDF" in str(x)]
mas1 = mas1[0]/ _astro_utilities.master_dir
print ("mas1: ", format(mas1))

DOINGDIRs = sorted([x for x in DOINGDIRs if "_LIGHT_" in str(x)])
print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

# filter_str = '2023-12-'
# DOINGDIRs = [x for x in DOINGDIRs if filter_str in x]
# remove = 'BIAS'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
# remove = 'DARK'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
# remove = 'FLAT'
# DOINGDIRs = [x for x in DOINGDIRs if remove not in x]
print ("DOINGDIRs: ", DOINGDIRs)
print ("len(DOINGDIRs): ", len(DOINGDIRs))
#######################################################

In [ ]:
for DOINGDIR in DOINGDIRs[3:4] :
    DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    if str(DOINGDIR.parts[-2]) == "RiLA600_STX-16803_-_1bin" :
        DOINGDIR = DOINGDIR / _astro_utilities.REDUC_nightsky_dir
    if str(DOINGDIR.parts[-2]) == "GSON300_STF-8300M_-_1bin" :
        DOINGDIR = DOINGDIR / _astro_utilities.reduced_dir
    
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        print(f"Starting: {str(DOINGDIR.parts[-1])}")

        summary = yfu.make_summary(DOINGDIR/"*.fit*")
        print("len(summary):", len(summary))
        print("summary:", summary)
        #print(summary["file"][0])
        df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
        df_light = df_light.reset_index(drop=True)
        print("df_light:\n{}".format(df_light))

        for _, row  in df_light.iterrows():

            fpath = Path(row["file"])
            hdul = fits.open(fpath)

## astrometry solving





### apstrometry 연결

nova.astrometry.net 으로부터 api key를 이용하여 연결합니다.


In [ ]:
ast = AstrometryNet()

# ger from nova.astrometry.net
ast.api_key = 'bldvwzzuvktnwfph' #must changed...

In [ ]:
# from astroquery.astrometry_net import AstrometryNet

# ast = AstrometryNet()

# # ger from nova.astrometry.net
# ast.api_key = 'bldvwzzuvktnwfph' #must changed...

# wcs_header = ast.solve_from_image(fpath)
# print("wcs_header :", wcs_header)

### file solving

아래와 같이 파일을 업로드하여 plate solvin을 해 볼 수 있다.

In [ ]:
# from astroquery.astrometry_net import AstrometryNet

# ast = AstrometryNet()

# # ger from nova.astrometry.net
# ast.api_key = 'bldvwzzuvktnwfph' #must changed...

# #fpath = Path('/mnt/Rdata/OBS_data/2024-EXO/GSON300_STF-8300M_-_1bin/Qatar-1b_LIGHT_-_2024-05-29_-_GSON300_STF-8300M_-_1bin/reduced/Qatar-1b_LIGHT_R_2024-05-29-12-06-44_180sec_GSON300_STF-8300M_20c_1bin.fit')

# submission_id = None
# solve_timeout = 600
# print(fpath)

# hdul = fits.open(str(fpath))
# if not 'B_1_1' in hdul[0].header :
#     try_again = True

# while try_again:
#     try:
#         if not submission_id:
#             wcs_header = ast.solve_from_image(str(fpath),
#                                 force_image_upload=True,
#                                 solve_timeout = solve_timeout,
#                                 submission_id=submission_id)
#         else:
#             wcs_header = ast.monitor_submission(submission_id,
#                                                 solve_timeout = solve_timeout)
#     except TimeoutError as e:
#         submission_id = e.args[1]
#     else:
#         # got a result, so terminate
#         try_again = False

# if wcs_header:
#     # Code to execute when solve succeeds
#     print("fits file solved successfully...")

#     with fits.open(str(fpath), mode='update') as hdul:
#         for card in wcs_header :
#             try: 
#                 print(card, wcs_header[card], wcs_header.comments[card])
#                 hdul[0].header.set(card, wcs_header[card], wcs_header.comments[card])
#             except : 
#                 print(card)
#         hdul.flush

#     print(str(fpath.parents[0] / f'{fpath.stem}.fits')+" is created...")
# else:
#     # Code to execute when solve fails
#     print("fits file solving failure...")

In [ ]:
for DOINGDIR in DOINGDIRs[:1] :
    DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    if str(DOINGDIR.parts[-2]) == "RiLA600_STX-16803_-_1bin" :
        DOINGDIR = DOINGDIR / _astro_utilities.REDUC_nightsky_dir
    if str(DOINGDIR.parts[-2]) == "GSON300_STF-8300M_-_1bin" :
        DOINGDIR = DOINGDIR / _astro_utilities.reduced_dir
    
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

    if len(fits_in_dir) == 0 :
        print(f"There is no fits fils in {DOINGDIR}")
        pass
    else : 
        print(f"Starting: {str(DOINGDIR.parts[-1])}")

        summary = yfu.make_summary(DOINGDIR/"*.fit*")
        print("len(summary):", len(summary))
        print("summary:", summary)
        #print(summary["file"][0])
        df_light = summary.loc[summary["IMAGETYP"] == "LIGHT"].copy()
        df_light = df_light.reset_index(drop=True)
        print("df_light:\n{}".format(df_light))

        for _, row  in df_light.iterrows():

            fpath = Path(row["file"])
            print(fpath)
            hdul = fits.open(fpath)

            submission_id = None
            solve_timeout = 600

            if 'PIXSCALE' in hdul[0].header:
                PIXc = hdul[0].header['PIXSCALE']
            else : 
                PIXc = _astro_utilities.calPixScale(hdul[0].header['FOCALLEN'], 
                                                    hdul[0].header['XPIXSZ'],
                                                    hdul[0].header['XBINNING'])
            print("PIXc : ", PIXc)
            hdul.close()

            SOLVE, ASTAP, LOCAL = _astro_utilities.checkPSolve(fpath)
            print("SOLVE:", SOLVE, "ASTAP:", ASTAP, "LOCAL:", LOCAL)

            if SOLVE :
                print(f"{fpath.name} is already solved...")
            else :             
                try_again = True                

                try : 
                    
                    while try_again:
                        try:
                            if not submission_id:
                                wcs_header = ast.solve_from_image(str(fpath),
                                                    force_image_upload=True,
                                                    solve_timeout = solve_timeout,
                                                    submission_id=submission_id)
                            else:
                                wcs_header = ast.monitor_submission(submission_id,
                                                                    solve_timeout = solve_timeout)
                        except TimeoutError as e:
                            submission_id = e.args[1]
                        else:
                            # got a result, so terminate
                            try_again = False

                    if not wcs_header:
                        # Code to execute when solve fails
                        print("fits file solving failure...")

                    else:
                        # Code to execute when solve succeeds
                        print("fits file solved successfully...")

                        with fits.open(str(fpath), mode='update') as hdul:
                            for card in wcs_header :
                                try: 
                                    print(card, wcs_header[card], wcs_header.comments[card])
                                    hdul[0].header.set(card, wcs_header[card], wcs_header.comments[card])
                                except : 
                                    print(card)
                            hdul.flush

                        print(str(fpath)+" is created...")
                
                except Exception as err: 
                    print("Err :", err)
                    continue
                